# Brain Tumor MRI Classification

A deep learning project for classifying brain MRI images into four categories: **glioma, meningioma, pituitary, and no tumor**.

> **Note:** This project is for educational/research purposes and is not a clinical diagnostic tool.


## 1. Dataset

The project uses the **Brain Tumor MRI Dataset** from Kaggle. The original dataset contains **7,033 MRI images**: 5,712 training images and 1,321 testing images across four classes.


In [ ]:
# Install dependencies in Google Colab if needed
!pip install -q opencv-python pillow pandas numpy matplotlib seaborn scikit-learn tqdm tensorflow


In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm


In [ ]:
# Update this path for your environment
DATA_DIR = "/content/brain-tumor-mri-dataset"
TRAIN_DIR = os.path.join(DATA_DIR, "Training")
TEST_DIR = os.path.join(DATA_DIR, "Testing")

if not os.path.isdir(TRAIN_DIR) or not os.path.isdir(TEST_DIR):
    raise FileNotFoundError("Set DATA_DIR to the folder containing Training/ and Testing/.")


## 2. Exploratory Data Analysis

The original project run found no corrupted images in the training or testing set.


In [ ]:
def count_images(directory):
    counts = {}
    for cls in sorted(os.listdir(directory)):
        class_dir = os.path.join(directory, cls)
        if os.path.isdir(class_dir):
            counts[cls] = len(os.listdir(class_dir))
    return counts

train_counts = count_images(TRAIN_DIR)
test_counts = count_images(TEST_DIR)
print("Training:", train_counts)
print("Testing:", test_counts)

plt.figure(figsize=(10, 5))
classes = sorted(set(train_counts) | set(test_counts))
x = np.arange(len(classes))
width = 0.38
plt.bar(x - width/2, [train_counts.get(c, 0) for c in classes], width, label="Training")
plt.bar(x + width/2, [test_counts.get(c, 0) for c in classes], width, label="Testing")
plt.xticks(x, classes, rotation=20)
plt.ylabel("Number of images")
plt.title("Dataset Class Distribution")
plt.legend()
plt.tight_layout()
plt.show()


## 3. Preprocessing

- Convert MRI images to grayscale
- Crop the brain region using contour detection
- Resize to **124 × 124**
- Normalize pixel values
- Repeat the grayscale channel three times for ImageNet-pretrained ResNet50


In [ ]:
IMG_SIZE = (124, 124)
PROCESSED_TRAIN_DIR = "/content/processed_Training_data"
PROCESSED_TEST_DIR = "/content/processed_Testing_data"

def crop_brain_contour(image):
    _, thresh = cv2.threshold(image, 5, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return image
    contour = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(contour)
    return image[y:y+h, x:x+w]

def process_dataset(input_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    records = []
    for class_name in sorted(os.listdir(input_dir)):
        class_dir = os.path.join(input_dir, class_name)
        if not os.path.isdir(class_dir):
            continue
        output_class_dir = os.path.join(output_dir, class_name)
        os.makedirs(output_class_dir, exist_ok=True)
        for filename in tqdm(os.listdir(class_dir), desc=f"Processing {class_name}"):
            input_path = os.path.join(class_dir, filename)
            image = cv2.imread(input_path, cv2.IMREAD_GRAYSCALE)
            if image is None:
                continue
            cropped = crop_brain_contour(image)
            resized = cv2.resize(cropped, IMG_SIZE)
            normalized = resized.astype(np.float32) / 255.0
            rgb = np.stack([normalized] * 3, axis=-1)
            base_name = os.path.splitext(filename)[0]
            output_path = os.path.join(output_class_dir, f"{base_name}_rgb.jpg")
            cv2.imwrite(output_path, (rgb * 255).astype(np.uint8))
            records.append([output_path, class_name])
    pd.DataFrame(records, columns=["image_path", "label"]).to_csv(os.path.join(output_dir, "images_with_labels.csv"), index=False)

process_dataset(TRAIN_DIR, PROCESSED_TRAIN_DIR)
process_dataset(TEST_DIR, PROCESSED_TEST_DIR)
print("Preprocessing complete.")


## 4. ResNet50 Transfer Learning

The final experiment uses **ResNet50 pretrained on ImageNet** with a custom classification head and regularization.


In [ ]:
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

BATCH_SIZE = 32
EPOCHS = 60
LEARNING_RATE = 1e-3

train_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2, rotation_range=15)
validation_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(PROCESSED_TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode="categorical", subset="training", seed=42)
validation_generator = validation_datagen.flow_from_directory(PROCESSED_TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode="categorical", subset="validation", shuffle=False, seed=42)
test_generator = test_datagen.flow_from_directory(PROCESSED_TEST_DIR, target_size=IMG_SIZE, batch_size=1, class_mode="categorical", shuffle=False)

base_model = ResNet50(weights="imagenet", include_top=False, input_shape=(124, 124, 3))
model = Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.5),
    layers.Dense(64, activation="relu", kernel_regularizer=l2(0.001)),
    layers.Dropout(0.5),
    layers.Dense(train_generator.num_classes, activation="softmax")
])

model.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss="categorical_crossentropy", metrics=["accuracy"])
callbacks = [
    EarlyStopping(monitor="val_accuracy", patience=10, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.1, patience=5, min_lr=1e-6)
]
model.summary()


In [ ]:
history = model.fit(train_generator, epochs=EPOCHS, validation_data=validation_generator, callbacks=callbacks, verbose=1)


## 5. Evaluation

The recorded final run achieved approximately **95% test accuracy** on **1,311 test images**.


In [ ]:
test_generator.reset()
y_prob = model.predict(test_generator, verbose=0)
y_pred = np.argmax(y_prob, axis=1)
y_true = test_generator.classes
class_labels = list(test_generator.class_indices.keys())

print(classification_report(y_true, y_pred, target_names=class_labels, digits=4))
print(f"Accuracy : {accuracy_score(y_true, y_pred):.4f}")
print(f"Precision: {precision_score(y_true, y_pred, average='weighted'):.4f}")
print(f"Recall   : {recall_score(y_true, y_pred, average='weighted'):.4f}")
print(f"F1 Score : {f1_score(y_true, y_pred, average='weighted'):.4f}")

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_labels, yticklabels=class_labels)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("ResNet50 Confusion Matrix")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and Validation Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
model.save("brain_tumor_resnet50.keras")


## 6. Sample Predictions

The original run correctly classified one sample from each of the four classes. See `results/` for the saved prediction images.


In [ ]:
import glob
from tensorflow.keras.preprocessing import image

for true_class in ["glioma", "meningioma", "notumor", "pituitary"]:
    paths = glob.glob(os.path.join(PROCESSED_TEST_DIR, true_class, "*.jpg"))
    if not paths:
        continue
    image_path = paths[0]
    img = image.load_img(image_path, target_size=IMG_SIZE)
    x = np.expand_dims(image.img_to_array(img) / 255.0, axis=0)
    probabilities = model.predict(x, verbose=0)[0]
    predicted_index = int(np.argmax(probabilities))
    predicted_class = class_labels[predicted_index]
    print(f"True: {true_class} | Predicted: {predicted_class} | Confidence: {probabilities[predicted_index]:.2%}")
    plt.figure(figsize=(4, 4))
    plt.imshow(img)
    plt.title(f"Predicted: {predicted_class} ({probabilities[predicted_index]:.2%})")
    plt.axis("off")
    plt.show()


## 7. Safety and Scope

This is a machine-learning classification experiment for educational/research use. It must not be used to diagnose patients, recommend treatment, or replace qualified medical professionals.
